# MentalDistress Phase 8 — All-Pair Specialists (GPU)

This notebook builds the complete bank of frozen-encoder pair specialists:

- **10 global-model seeds × 10 unordered label pairs = 100 heads**
- each seed uses its **own matching MentalRoBERTa encoder**;
- the encoder is frozen and the feature is the final-layer `<s>` representation;
- every specialist is a class-weighted `Linear(hidden_size, 2)` classifier;
- only locked training and validation data are used;
- the combined dataset and test set are explicitly forbidden;
- **Phase 8 performs no routing and no `tau` selection**.

Phase 9 will load these frozen heads, activate the three Phase 5 hard-pair types
(`K=3`), and calibrate the routing threshold on validation data without retraining
specialists. Test data remains untouched.

## Kaggle setup

Attach these three inputs and enable a GPU accelerator:

1. the locked MentalDistress dataset (train and validation CSVs);
2. the full ten-seed MentalRoBERTa output containing canonical run directories;
3. the Phase 5 bootstrap-stability results.

Internet access is unnecessary. Input paths are discovered beneath `/kaggle/input`
and validated by identity, hashes, manifests, seeds, run IDs, and label order.

Runtime is dominated by ten encoder passes over train and validation. Only one
encoder is loaded at a time, and completed seeds can be resumed safely within the
same Kaggle working session.


## 1. Imports and frozen configuration

Change only operational batch sizes if GPU memory requires it. Scientific training
settings, seeds, labels, hashes, and the 100-head target are frozen.


In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import gc
import hashlib
import itertools
import json
import math
import random
import re
import shutil
import tempfile
import time
import zipfile
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from safetensors.torch import load_file, save_file
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

EXPECTED_SEEDS = [13, 42, 77, 123, 211, 340, 512, 1024, 2026, 3407]
LABELS = ["Suicidal", "Depressed", "Anxious", "Frustrated", "Others"]
LABEL2ID = {label: index for index, label in enumerate(LABELS)}
ALL_PAIRS = list(itertools.combinations(LABELS, 2))
EXPECTED_H = [
    "Frustrated|Others",
    "Suicidal|Depressed",
    "Depressed|Others",
]

TRAIN_ROWS = 6954
VALIDATION_ROWS = 1490
TRAIN_SHA256 = "1bac0d2f4e0f4d3da940b08a21bd3c6e18b1d958c57cb263c89a900381572d5b"
VALIDATION_SHA256 = "88835cd714b02efbd230536c7de3f455e807051a23a1786640b0e8b26b430165"
GLOBAL_MODEL_KEY = "mentalroberta"
GLOBAL_MODEL_SOURCE = "mental/mental-roberta-base"
GLOBAL_MODEL_REVISION = "610e9d35d8f2550d713579077242d5e523209cea"

MAX_LENGTH = 512
ENCODER_BATCH_SIZE = 16
HEAD_BATCH_SIZE = 64
HEAD_MAX_EPOCHS = 60
HEAD_LR = 1e-3
HEAD_WEIGHT_DECAY = 1e-2
HEAD_PATIENCE = 6
HEAD_MAX_GRAD_NORM = 1.0
MASTER_HEAD_SEED = 20260822
GLOBAL_LOGIT_ATOL = 1e-5
GLOBAL_LOGIT_RTOL = 1e-6
RESUME_COMPLETED_SEEDS = True

HEAD_PROTOCOL = {
    "phase": 8,
    "seeds": EXPECTED_SEEDS,
    "labels": LABELS,
    "pairs": [f"{a}|{b}" for a, b in ALL_PAIRS],
    "representation": "final_hidden_state_token_0",
    "max_length": MAX_LENGTH,
    "head_architecture": "Linear(hidden_size, 2)",
    "loss": "class_weighted_cross_entropy_N_over_2Nc",
    "head_batch_size": HEAD_BATCH_SIZE,
    "maximum_epochs": HEAD_MAX_EPOCHS,
    "learning_rate": HEAD_LR,
    "weight_decay": HEAD_WEIGHT_DECAY,
    "patience": HEAD_PATIENCE,
    "maximum_gradient_norm": HEAD_MAX_GRAD_NORM,
    "master_head_seed": MASTER_HEAD_SEED,
    "train_sha256": TRAIN_SHA256,
    "validation_sha256": VALIDATION_SHA256,
    "global_model_key": GLOBAL_MODEL_KEY,
    "global_model_source": GLOBAL_MODEL_SOURCE,
    "global_model_revision": GLOBAL_MODEL_REVISION,
    "encoder_batch_size": ENCODER_BATCH_SIZE,
    "global_logit_atol": GLOBAL_LOGIT_ATOL,
    "global_logit_rtol": GLOBAL_LOGIT_RTOL,
}
PROTOCOL_FINGERPRINT = hashlib.sha256(
    json.dumps(HEAD_PROTOCOL, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
OUTPUT_ROOT = WORK_ROOT / "phase8_all_pair_specialists_results"
ZIP_PATH = WORK_ROOT / "MentalDistress_Phase8_All_Pair_Specialists_Results.zip"
ZIP_HASH_PATH = ZIP_PATH.with_suffix(".zip.sha256")

FORBIDDEN_INPUT_NAMES = {
    "MentalDistress_LOCKED.csv",
    "MentalDistress_test_LOCKED.csv",
    "test.csv",
}

def require_cuda() -> torch.device:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "Phase 8 requires a Kaggle GPU. Enable Accelerator -> GPU and restart the session."
        )
    device = torch.device("cuda")
    print("CUDA device:", torch.cuda.get_device_name(0))
    print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
    return device

DEVICE = require_cuda()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for stale_archive in [ZIP_PATH, ZIP_HASH_PATH]:
    if stale_archive.exists():
        stale_archive.unlink()
print("Expected heads:", len(EXPECTED_SEEDS), "seeds ×", len(ALL_PAIRS), "pairs =", 100)


CUDA device: Tesla T4
PyTorch: 2.10.0+cu128 CUDA runtime: 12.8
Expected heads: 10 seeds × 10 pairs = 100


## 2. Integrity, discovery, metrics, and deterministic packaging helpers

Manifest entries are resolved defensively and may not escape their declared root.
Discovery never opens a combined or test dataset.


In [2]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def write_json(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(json_ready(payload), indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def safe_manifest_target(root: Path, relative_path: str) -> Path:
    posix = PurePosixPath(str(relative_path))
    if posix.is_absolute() or ".." in posix.parts:
        raise ValueError(f"Unsafe manifest path: {relative_path!r}")
    root_resolved = root.resolve()
    candidate = (root / Path(*posix.parts)).resolve()
    try:
        candidate.relative_to(root_resolved)
    except ValueError as exc:
        raise ValueError(f"Manifest path escapes root: {relative_path!r}") from exc
    return candidate


def verify_manifest(manifest_path: Path) -> dict:
    manifest_path = manifest_path.resolve()
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Missing manifest: {manifest_path}")
    frame = pd.read_csv(manifest_path, dtype={"path": str, "sha256": str})
    required = {"path", "sha256", "bytes"}
    if not required.issubset(frame.columns):
        raise ValueError(f"Manifest schema mismatch: {manifest_path}")
    if frame["path"].duplicated().any():
        raise ValueError(f"Duplicate manifest paths: {manifest_path}")
    listed_paths = set(frame["path"].astype(str))
    actual_paths = {
        path.relative_to(manifest_path.parent).as_posix()
        for path in manifest_path.parent.rglob("*")
        if path.is_file() and path != manifest_path
    }
    if listed_paths != actual_paths:
        missing = sorted(actual_paths - listed_paths)
        extra = sorted(listed_paths - actual_paths)
        raise ValueError(
            f"Manifest membership mismatch: {manifest_path}; "
            f"unlisted={missing[:5]}, nonexistent={extra[:5]}"
        )
    checked = 0
    for row in frame.itertuples(index=False):
        target = safe_manifest_target(manifest_path.parent, row.path)
        if not target.is_file():
            raise FileNotFoundError(f"Manifest target missing: {target}")
        if target.stat().st_size != int(row.bytes):
            raise ValueError(f"Byte-size mismatch: {target}")
        if sha256_file(target) != str(row.sha256).lower():
            raise ValueError(f"SHA-256 mismatch: {target}")
        checked += 1
    return {
        "manifest": str(manifest_path),
        "manifest_sha256": sha256_file(manifest_path),
        "files_verified": checked,
        "status": "PASS",
    }


def assert_allowed_input_path(path: Path) -> None:
    if path.name in FORBIDDEN_INPUT_NAMES:
        raise RuntimeError(f"Forbidden dataset selected: {path}")
    lowered_parts = [part.lower() for part in path.parts]
    if any(part in {"test", "testing", "test-set", "test_set"} for part in lowered_parts):
        raise RuntimeError(f"Forbidden test path selected: {path}")


def is_explicit_false(value) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return not bool(value)
    return isinstance(value, str) and value.strip().lower() == "false"


def choose_identical_file(paths: list[Path], expected_hash: str | None = None) -> Path:
    paths = sorted({path.resolve() for path in paths}, key=lambda p: (len(p.parts), str(p)))
    if not paths:
        raise FileNotFoundError("No candidate files")
    hashes = {path: sha256_file(path) for path in paths}
    if expected_hash is not None:
        paths = [path for path in paths if hashes[path] == expected_hash]
        if not paths:
            raise ValueError(f"No candidate matches expected SHA-256 {expected_hash}")
    elif len(set(hashes.values())) != 1:
        raise ValueError("Ambiguous candidate files have different hashes")
    chosen = paths[0]
    assert_allowed_input_path(chosen)
    return chosen


def discover_locked_csv(filename: str, expected_hash: str) -> Path:
    candidates = [path for path in INPUT_ROOT.rglob(filename) if path.is_file()]
    return choose_identical_file(candidates, expected_hash)


def discover_phase5() -> tuple[Path, dict, dict]:
    candidates = [
        path for path in INPUT_ROOT.rglob("phase5_hard_pair_selection.json") if path.is_file()
    ]
    valid = []
    for selection_path in candidates:
        phase5_root = selection_path.parent
        if (phase5_root / "phase5_protocol.json").is_file() and (
            phase5_root / "sha256_manifest.csv"
        ).is_file():
            valid.append(selection_path)
    selection_path = choose_identical_file(valid)
    phase5_root = selection_path.parent
    manifest_result = verify_manifest(phase5_root / "sha256_manifest.csv")
    selection = json.loads(selection_path.read_text(encoding="utf-8"))
    protocol = json.loads((phase5_root / "phase5_protocol.json").read_text(encoding="utf-8"))
    if selection.get("status") != "FROZEN":
        raise ValueError("Phase 5 hard-pair selection is not FROZEN")
    if selection.get("H") != EXPECTED_H or int(selection.get("K", -1)) != 3:
        raise ValueError("Phase 5 H/K do not match the frozen project decision")
    if not is_explicit_false(selection.get("test_accessed")) or not is_explicit_false(
        protocol.get("test_accessed")
    ):
        raise ValueError("Phase 5 reports test access")
    return phase5_root, selection, manifest_result


def discover_seed_summary() -> tuple[Path, pd.DataFrame]:
    candidates = [
        path for path in INPUT_ROOT.rglob("mentalroberta_full_10seed_summary.csv") if path.is_file()
    ]
    if not candidates:
        raise FileNotFoundError("mentalroberta_full_10seed_summary.csv was not found")
    valid = []
    for path in candidates:
        frame = pd.read_csv(path)
        if {"seed", "run_id", "status", "test_accessed"}.issubset(frame.columns):
            seeds = sorted(frame["seed"].astype(int).tolist())
            if seeds == sorted(EXPECTED_SEEDS):
                valid.append(path)
    summary_path = choose_identical_file(valid)
    frame = pd.read_csv(summary_path).sort_values("seed").reset_index(drop=True)
    if frame["seed"].astype(int).duplicated().any() or len(frame) != 10:
        raise ValueError("Ten-seed summary must contain one row per expected seed")
    if not frame["status"].astype(str).str.upper().eq("PASS").all():
        raise ValueError("A global-model seed does not have PASS status")
    if not frame["test_accessed"].map(is_explicit_false).all():
        raise ValueError("Ten-seed summary test_accessed must be explicitly false for every seed")
    return summary_path, frame


def resolve_checkpoint_dir(run_dir: Path) -> Path:
    checkpoint_dir = run_dir / "models" / "best_model"
    required = [
        checkpoint_dir / "config.json",
        checkpoint_dir / "model.safetensors",
        checkpoint_dir / "tokenizer_config.json",
    ]
    if not all(path.is_file() for path in required):
        missing = [str(path.relative_to(run_dir)) for path in required if not path.is_file()]
        raise FileNotFoundError(f"Incomplete best_model checkpoint in {run_dir}: {missing}")
    has_tokenizer_payload = (checkpoint_dir / "tokenizer.json").is_file() or all(
        (checkpoint_dir / name).is_file() for name in ["vocab.json", "merges.txt"]
    )
    if not has_tokenizer_payload:
        raise FileNotFoundError(f"Tokenizer payload missing from {checkpoint_dir}")
    return checkpoint_dir


def discover_canonical_runs(summary: pd.DataFrame) -> dict[int, dict]:
    discovered = {}
    for row in summary.itertuples(index=False):
        seed = int(row.seed)
        run_id = str(row.run_id)
        candidates = []
        for path in INPUT_ROOT.rglob(run_id):
            if path.is_dir() and path.name == run_id:
                if (path / "models").is_dir() and (path / "artifacts").is_dir():
                    candidates.append(path.resolve())
        candidates = sorted(set(candidates), key=lambda p: (len(p.parts), str(p)))
        if not candidates:
            raise FileNotFoundError(f"Canonical run directory not found for seed {seed}: {run_id}")
        candidate_records = []
        for candidate in candidates:
            models_root = candidate / "models"
            artifact_dir = candidate / "artifacts"
            model_dir = resolve_checkpoint_dir(candidate)
            model_manifest = verify_manifest(models_root / "sha256_manifest.csv")
            artifact_manifest = verify_manifest(artifact_dir / "sha256_manifest.csv")
            candidate_records.append(
                {
                    "run_dir": candidate,
                    "models_root": models_root,
                    "model_dir": model_dir,
                    "artifact_dir": artifact_dir,
                    "model_manifest": model_manifest,
                    "artifact_manifest": artifact_manifest,
                }
            )
        identities = {
            (record["model_manifest"]["manifest_sha256"], record["artifact_manifest"]["manifest_sha256"])
            for record in candidate_records
        }
        if len(identities) != 1:
            raise ValueError(f"Duplicate run directories differ for seed {seed}: {run_id}")
        record = candidate_records[0]
        run_dir = record["run_dir"]
        models_root = record["models_root"]
        model_dir = record["model_dir"]
        artifact_dir = record["artifact_dir"]
        model_manifest = record["model_manifest"]
        artifact_manifest = record["artifact_manifest"]
        model_sha256 = sha256_file(model_dir / "model.safetensors")
        config = json.loads((artifact_dir / "config.json").read_text(encoding="utf-8"))
        integrity = json.loads((artifact_dir / "dataset_integrity.json").read_text(encoding="utf-8"))
        if int(config.get("seed", -1)) != seed or config.get("run_id") != run_id:
            raise ValueError(f"Run identity mismatch for seed {seed}")
        if not is_explicit_false(config.get("test_accessed")) or not is_explicit_false(
            integrity.get("test_accessed")
        ):
            raise ValueError(f"Run {run_id} reports test access")
        if config.get("id2label") != {str(i): label for i, label in enumerate(LABELS)}:
            raise ValueError(f"Label order mismatch for seed {seed}")
        expected_run_prefix = f"mentalroberta_full_seed{seed}_len512_"
        if not re.fullmatch(re.escape(expected_run_prefix) + r"[0-9a-f]{12}", run_id):
            raise ValueError(f"Unexpected run identifier for seed {seed}: {run_id}")
        if config.get("run_mode") != "full" or int(config.get("max_length", -1)) != MAX_LENGTH:
            raise ValueError(f"Run mode/length mismatch for seed {seed}")
        if int(config.get("train_rows_used", -1)) != TRAIN_ROWS or int(
            config.get("validation_rows_used", -1)
        ) != VALIDATION_ROWS:
            raise ValueError(f"Run row-count metadata mismatch for seed {seed}")
        model_config = config.get("model", {})
        if (
            model_config.get("model_key") != GLOBAL_MODEL_KEY
            or model_config.get("source") != GLOBAL_MODEL_SOURCE
            or model_config.get("revision") != GLOBAL_MODEL_REVISION
        ):
            raise ValueError(f"Model family/revision mismatch for seed {seed}")
        if integrity.get("rows") != {"train": TRAIN_ROWS, "validation": VALIDATION_ROWS}:
            raise ValueError(f"Dataset row metadata mismatch for seed {seed}")
        if integrity.get("sha256", {}).get("train") != TRAIN_SHA256:
            raise ValueError(f"Training hash mismatch in seed {seed} artifacts")
        if integrity.get("sha256", {}).get("validation") != VALIDATION_SHA256:
            raise ValueError(f"Validation hash mismatch in seed {seed} artifacts")
        discovered[seed] = {
            "seed": seed,
            "run_id": run_id,
            "run_dir": run_dir,
            "models_root": models_root,
            "model_dir": model_dir,
            "model_sha256": model_sha256,
            "artifact_dir": artifact_dir,
            "model_manifest": model_manifest,
            "artifact_manifest": artifact_manifest,
            "model_manifest_sha256": model_manifest["manifest_sha256"],
            "artifact_manifest_sha256": artifact_manifest["manifest_sha256"],
        }
    if sorted(discovered) != sorted(EXPECTED_SEEDS):
        raise ValueError("Canonical run discovery did not produce exactly ten expected seeds")
    return discovered


def compute_class_weights(binary_targets) -> torch.Tensor:
    targets = np.asarray(binary_targets, dtype=np.int64)
    counts = np.bincount(targets, minlength=2)
    if targets.size == 0 or np.any(counts == 0):
        raise ValueError("Both pair classes must be present")
    return torch.tensor(targets.size / (2.0 * counts), dtype=torch.float32)


def binary_metrics(y_true, y_pred) -> dict:
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1], zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "class0_precision": float(precision[0]),
        "class0_recall": float(recall[0]),
        "class0_f1": float(f1[0]),
        "class0_support": int(support[0]),
        "class1_precision": float(precision[1]),
        "class1_recall": float(recall[1]),
        "class1_f1": float(f1[1]),
        "class1_support": int(support[1]),
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
    }


def deterministic_head_seed(model_seed: int, pair_index: int) -> int:
    state = np.random.SeedSequence(
        [MASTER_HEAD_SEED, int(model_seed), int(pair_index)]
    ).generate_state(1, dtype=np.uint32)
    return int(state[0])


def selection_key(macro_f1: float, validation_loss: float, epoch: int) -> tuple:
    return (-float(macro_f1), float(validation_loss), int(epoch))


def pair_slug(class_a: str, class_b: str) -> str:
    return f"{class_a.lower()}__{class_b.lower()}"


def expected_seed_pair_combinations() -> set[tuple[int, str]]:
    return {
        (seed, f"{class_a}|{class_b}")
        for seed in EXPECTED_SEEDS
        for class_a, class_b in ALL_PAIRS
    }


def expected_seed_relative_paths() -> set[str]:
    expected = {
        "seed_complete.json",
        "seed_head_metrics.csv",
        "seed_validation_predictions.csv",
        "seed_training_history.csv",
    }
    for class_a, class_b in ALL_PAIRS:
        slug = pair_slug(class_a, class_b)
        expected.update(
            {
                f"{slug}/head.safetensors",
                f"{slug}/metadata.json",
                f"{slug}/metrics.json",
                f"{slug}/training_history.csv",
                f"{slug}/validation_predictions.csv",
            }
        )
    return expected


def expected_output_relative_paths(include_manifest: bool) -> set[str]:
    expected = {
        "phase8_input_inventory.csv",
        "phase8_head_metrics.csv",
        "phase8_validation_predictions.csv",
        "phase8_training_history.csv",
        "phase8_global_reproduction.csv",
        "phase8_completeness_matrix.csv",
        "phase8_pair_summary.csv",
        "phase8_seed_summary.csv",
        "phase8_protocol.json",
        "phase8_report.md",
        "phase8_execution_summary.json",
    }
    for seed in EXPECTED_SEEDS:
        expected.update(
            f"heads/seed_{seed}/{relative}" for relative in expected_seed_relative_paths()
        )
    if include_manifest:
        expected.add("sha256_manifest.csv")
    return expected


def build_deterministic_zip(source_dir: Path, target_zip: Path) -> str:
    source_dir = source_dir.resolve()
    target_zip.parent.mkdir(parents=True, exist_ok=True)
    temporary = target_zip.with_suffix(target_zip.suffix + ".tmp")
    if temporary.exists():
        temporary.unlink()
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as archive:
        for path in sorted(p for p in source_dir.rglob("*") if p.is_file()):
            relative = path.relative_to(source_dir)
            archive_name = (Path(source_dir.name) / relative).as_posix()
            info = zipfile.ZipInfo(archive_name, date_time=(2020, 1, 1, 0, 0, 0))
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = 0o100644 << 16
            archive.writestr(info, path.read_bytes(), compress_type=zipfile.ZIP_DEFLATED, compresslevel=9)
    temporary.replace(target_zip)
    return sha256_file(target_zip)


def write_output_manifest(root: Path) -> pd.DataFrame:
    manifest_path = root / "sha256_manifest.csv"
    if manifest_path.exists():
        manifest_path.unlink()
    rows = []
    for path in sorted(p for p in root.rglob("*") if p.is_file()):
        rows.append(
            {
                "path": path.relative_to(root).as_posix(),
                "sha256": sha256_file(path),
                "bytes": path.stat().st_size,
            }
        )
    frame = pd.DataFrame(rows, columns=["path", "sha256", "bytes"])
    frame.to_csv(manifest_path, index=False)
    return frame


## 3. Lightweight self-tests

These tests execute before input discovery or model loading. They check the most
error-prone deterministic and safety helpers without touching scientific outputs.


In [3]:
assert len(ALL_PAIRS) == 10
assert len(set(ALL_PAIRS)) == 10
assert ALL_PAIRS[0] == ("Suicidal", "Depressed")
assert ALL_PAIRS[-1] == ("Frustrated", "Others")

_weights = compute_class_weights([0, 0, 0, 1])
np.testing.assert_allclose(_weights.numpy(), [4 / 6, 2.0], rtol=0, atol=1e-7)

assert deterministic_head_seed(42, 0) == deterministic_head_seed(42, 0)
assert deterministic_head_seed(42, 0) != deterministic_head_seed(42, 1)
assert selection_key(0.9, 0.5, 4) < selection_key(0.8, 0.1, 1)
assert selection_key(0.9, 0.4, 5) < selection_key(0.9, 0.5, 2)
assert selection_key(0.9, 0.4, 2) < selection_key(0.9, 0.4, 3)

_perfect = binary_metrics([0, 0, 1, 1], [0, 0, 1, 1])
assert _perfect["macro_f1"] == 1.0 and _perfect["balanced_accuracy"] == 1.0

with tempfile.TemporaryDirectory() as _tmp:
    _tmp = Path(_tmp)
    (_tmp / "source").mkdir()
    (_tmp / "source" / "a.txt").write_text("alpha\n", encoding="utf-8")
    (_tmp / "source" / "b.txt").write_text("beta\n", encoding="utf-8")
    _hash1 = build_deterministic_zip(_tmp / "source", _tmp / "one.zip")
    _hash2 = build_deterministic_zip(_tmp / "source", _tmp / "two.zip")
    assert _hash1 == _hash2
    try:
        safe_manifest_target(_tmp / "source", "../escape.txt")
        raise AssertionError("Unsafe manifest path was accepted")
    except ValueError:
        pass

    _checkpoint = _tmp / "run" / "models" / "best_model"
    _checkpoint.mkdir(parents=True)
    for _name in [
        "config.json", "model.safetensors", "tokenizer_config.json", "vocab.json", "merges.txt"
    ]:
        (_checkpoint / _name).write_bytes(b"fixture")
    assert resolve_checkpoint_dir(_tmp / "run") == _checkpoint

    _manifest_root = _tmp / "manifest_root"
    _manifest_root.mkdir()
    (_manifest_root / "payload.txt").write_text("verified\n", encoding="utf-8")
    _manifest = pd.DataFrame(
        [
            {
                "path": "payload.txt",
                "sha256": sha256_file(_manifest_root / "payload.txt"),
                "bytes": (_manifest_root / "payload.txt").stat().st_size,
            }
        ]
    )
    _manifest.to_csv(_manifest_root / "sha256_manifest.csv", index=False)
    assert verify_manifest(_manifest_root / "sha256_manifest.csv")["status"] == "PASS"
    (_manifest_root / "unlisted.txt").write_text("reject\n", encoding="utf-8")
    try:
        verify_manifest(_manifest_root / "sha256_manifest.csv")
        raise AssertionError("Incomplete manifest membership was accepted")
    except ValueError:
        pass

assert len(expected_seed_pair_combinations()) == 100
assert len(expected_seed_relative_paths()) == 54
assert len(expected_output_relative_paths(include_manifest=False)) == 551

print("PASS: lightweight Phase 8 helper tests")


PASS: lightweight Phase 8 helper tests


## 4. Discover and verify all inputs

This cell verifies the Phase 5 output manifest and every canonical seed's model and
artifact manifests. It selects only the exact locked train and validation files.


In [4]:
if not INPUT_ROOT.is_dir():
    raise FileNotFoundError("/kaggle/input is unavailable; run this notebook on Kaggle")

existing_output_files = {
    path.relative_to(OUTPUT_ROOT).as_posix()
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
}
unexpected_existing_outputs = existing_output_files - expected_output_relative_paths(
    include_manifest=True
)
if unexpected_existing_outputs:
    raise RuntimeError(
        "Unexpected stale files already exist under the Phase 8 output root. "
        f"Remove them before running: {sorted(unexpected_existing_outputs)[:10]}"
    )

TRAIN_PATH = discover_locked_csv("MentalDistress_train_LOCKED.csv", TRAIN_SHA256)
VALIDATION_PATH = discover_locked_csv(
    "MentalDistress_validation_LOCKED.csv", VALIDATION_SHA256
)
PHASE5_ROOT, PHASE5_SELECTION, PHASE5_MANIFEST_RESULT = discover_phase5()
SUMMARY_PATH, GLOBAL_SUMMARY = discover_seed_summary()
RUNS = discover_canonical_runs(GLOBAL_SUMMARY)

selected_input_paths = [TRAIN_PATH, VALIDATION_PATH, PHASE5_ROOT, SUMMARY_PATH]
selected_input_paths.extend(RUNS[seed]["run_dir"] for seed in EXPECTED_SEEDS)
for selected_path in selected_input_paths:
    assert_allowed_input_path(Path(selected_path))

inventory_rows = [
    {
        "role": "train_csv",
        "seed": "",
        "logical_source": TRAIN_PATH.name,
        "sha256": TRAIN_SHA256,
    },
    {
        "role": "validation_csv",
        "seed": "",
        "logical_source": VALIDATION_PATH.name,
        "sha256": VALIDATION_SHA256,
    },
    {
        "role": "phase5_results",
        "seed": "",
        "logical_source": PHASE5_ROOT.name,
        "sha256": PHASE5_MANIFEST_RESULT["manifest_sha256"],
    },
    {
        "role": "ten_seed_summary",
        "seed": "",
        "logical_source": SUMMARY_PATH.name,
        "sha256": sha256_file(SUMMARY_PATH),
    },
]
for seed in EXPECTED_SEEDS:
    inventory_rows.append(
        {
            "role": "canonical_global_run",
            "seed": seed,
            "logical_source": RUNS[seed]["run_id"],
            "sha256": RUNS[seed]["model_sha256"],
            "model_manifest_sha256": RUNS[seed]["model_manifest_sha256"],
            "artifact_manifest_sha256": RUNS[seed]["artifact_manifest_sha256"],
        }
    )
input_inventory = pd.DataFrame(inventory_rows)
input_inventory.to_csv(OUTPUT_ROOT / "phase8_input_inventory.csv", index=False)

print("Train:", TRAIN_PATH)
print("Validation:", VALIDATION_PATH)
print("Phase 5:", PHASE5_ROOT)
print("Canonical global runs:", len(RUNS))
display(input_inventory)


Train: /kaggle/input/datasets/sayemahmedshayeed/mentaldistress-locked-dataset/MentalDistress_train_LOCKED.csv
Validation: /kaggle/input/datasets/sayemahmedshayeed/mentaldistress-locked-dataset/MentalDistress_validation_LOCKED.csv
Phase 5: /kaggle/input/datasets/sumitnath07/mentaldistress-phase5-bootstrap-stability-results/phase5_bootstrap_stability_results
Canonical global runs: 10


,role,seed,logical_source,sha256,model_manifest_sha256,artifact_manifest_sha256
0,train_csv,,MentalDistress_train_LOCKED.csv,1bac0d2f4e0f4d3da940b08a21bd3c6e18b1d958c57cb2...,NaN,NaN
1,validation_csv,,MentalDistress_validation_LOCKED.csv,88835cd714b02efbd230536c7de3f455e807051a23a178...,NaN,NaN
2,phase5_results,,phase5_bootstrap_stability_results,6f94752ce77e95f740a14f774cb0665442a1bed844e829...,NaN,NaN
3,ten_seed_summary,,mentalroberta_full_10seed_summary.csv,9eaad5a54f19a396a4145a6d8b28269cfd855afbdbf9b9...,NaN,NaN
4,canonical_global_run,13,mentalroberta_full_seed13_len512_fea2991778f3,f504f1b5245bd4be50cfa05f4ddcd0c3ce0907a1b3d7b2...,160fbd297f6a1c3a23885bbcd303e1af91ba60994b9a66...,3022ffc1e9e38bc7854cd72b57335ea010c01553e72e05...
5,canonical_global_run,42,mentalroberta_full_seed42_len512_be72f916ca15,48a4e4d13f79b101eddb4ffaecc060cbe15be19ac8116a...,46d94a50380af739626e892f5469add55e0a7af70838ce...,525352cf34de51138d0d9ee821e32e86c422fceb0492da...
6,canonical_global_run,77,mentalroberta_full_seed77_len512_d14375990239,1d8782df9e618890fc8aa60abfc4a8ac45ef37eb9a3072...,a9ed78d37da58e2fd762d22c20e274abc04988e3f124ff...,96ae015d5568acae5757348e4c947a794d2d3701d0f24b...
7,canonical_global_run,123,mentalroberta_full_seed123_len512_d1ad313ee75d,9eae1838a91bf8d6eb71e754a2e8355610ec0e08293e89...,5dbb6873f4a1c1c921f736e60c5c90e31ef1b335871a0a...,2da945ab09582cb883bb9405a13d4cdd2aa8bab5799eeb...
8,canonical_global_run,211,mentalroberta_full_seed211_len512_149308113b40,e0678fc6669d6235fbd8700fb6f2ca2b72362c2e074dd5...,2f9aa82ebb059141e7882f4cc0773b5337fe42dba9ca1f...,a019e918b69ffd9b8a056fbcac170f3274e050d572302c...
9,canonical_global_run,340,mentalroberta_full_seed340_len512_6353292c8230,9757ed2b01cc708be9bf990e6a852a8a3cbdd1caba7f5c...,03c0690ee935b97125af69b615d8b67e1880bddf5d09b1...,9813993c77239d89c4e3311bfd3afae309965460c518f3...


## 5. Load the locked development splits

Row identity, label vocabulary, split disjointness, file hashes, and Phase 5 `H/K`
are hard gates. No combined or test file is read.


In [5]:
train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VALIDATION_PATH)

required_columns = {"original_row", "text", "label", "text_length"}
for split_name, frame, expected_rows in [
    ("train", train_df, TRAIN_ROWS),
    ("validation", validation_df, VALIDATION_ROWS),
]:
    if len(frame) != expected_rows:
        raise ValueError(f"{split_name} row count mismatch: {len(frame)} != {expected_rows}")
    if not required_columns.issubset(frame.columns):
        raise ValueError(f"{split_name} schema mismatch")
    if frame["original_row"].duplicated().any():
        raise ValueError(f"Duplicate original_row values in {split_name}")
    if frame[["original_row", "text", "label"]].isna().any().any():
        raise ValueError(f"Missing required values in {split_name}")
    if set(frame["label"].astype(str)) != set(LABELS):
        raise ValueError(f"Label vocabulary mismatch in {split_name}")

if set(train_df["original_row"]) & set(validation_df["original_row"]):
    raise ValueError("Locked train and validation row IDs overlap")
if sha256_file(TRAIN_PATH) != TRAIN_SHA256 or sha256_file(VALIDATION_PATH) != VALIDATION_SHA256:
    raise ValueError("Locked dataset hashes changed after discovery")
if PHASE5_SELECTION["H"] != EXPECTED_H or PHASE5_SELECTION["K"] != 3:
    raise ValueError("Frozen Phase 5 H/K changed")

train_df["text"] = train_df["text"].astype(str)
validation_df["text"] = validation_df["text"].astype(str)
train_df["label"] = train_df["label"].astype(str)
validation_df["label"] = validation_df["label"].astype(str)

print("PASS: locked development data and Phase 5 H/K verified")
print("Train rows:", len(train_df), "Validation rows:", len(validation_df))
display(train_df["label"].value_counts().reindex(LABELS).rename("train_count").to_frame())


PASS: locked development data and Phase 5 H/K verified
Train rows: 6954 Validation rows: 1490


,train_count
label,
Suicidal,1492
Depressed,1021
Anxious,1370
Frustrated,1292
Others,1779


## 6. Seed-matched encoder extraction

Validation logits are reproduced while the final-layer `<s>` features are extracted.
This proves that each specialist bank is attached to the intended global checkpoint.


In [6]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=False)


def load_seed_model(model_dir: Path):
    tokenizer = AutoTokenizer.from_pretrained(
        model_dir, local_files_only=True, use_fast=True
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        model_dir,
        local_files_only=True,
        output_hidden_states=True,
    )
    resolved_labels = [str(model.config.id2label[i]) for i in range(len(LABELS))]
    if resolved_labels != LABELS:
        raise ValueError(f"Checkpoint label order mismatch: {resolved_labels}")
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    model.eval().to(DEVICE)
    return tokenizer, model


@torch.inference_mode()
def extract_features_and_logits(texts, tokenizer, model) -> tuple[torch.Tensor, np.ndarray]:
    feature_chunks = []
    logit_chunks = []
    texts = list(texts)
    for start in range(0, len(texts), ENCODER_BATCH_SIZE):
        batch_texts = texts[start : start + ENCODER_BATCH_SIZE]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
        outputs = model(
            **encoded,
            output_hidden_states=True,
            return_dict=True,
        )
        feature_chunks.append(outputs.hidden_states[-1][:, 0, :].float().cpu())
        logit_chunks.append(outputs.logits.float().cpu().numpy())
    features = torch.cat(feature_chunks, dim=0).contiguous()
    logits = np.concatenate(logit_chunks, axis=0)
    if not torch.isfinite(features).all() or not np.isfinite(logits).all():
        raise ValueError("Non-finite encoder output")
    return features, logits


def verify_global_validation_artifact(
    seed: int,
    validation_frame: pd.DataFrame,
    reproduced_logits: np.ndarray,
    artifact_dir: Path,
    summary_row: pd.Series,
) -> dict:
    artifact = pd.read_csv(artifact_dir / "validation_predictions.csv")
    required_columns = {
        "original_row", "true_label", "true_id", "predicted_id", "predicted_label",
        *{f"logit_{label}" for label in LABELS},
        *{f"prob_{label}" for label in LABELS},
    }
    if not required_columns.issubset(artifact.columns):
        raise ValueError(f"Seed {seed}: validation artifact schema mismatch")
    if len(artifact) != len(validation_frame):
        raise ValueError(f"Seed {seed}: validation artifact row count mismatch")
    if artifact["original_row"].duplicated().any():
        raise ValueError(f"Seed {seed}: duplicate artifact row IDs")
    artifact = artifact.set_index("original_row").reindex(validation_frame["original_row"]).reset_index()
    if artifact.isna().any().any():
        raise ValueError(f"Seed {seed}: validation artifact row alignment failed")
    if not np.array_equal(artifact["true_label"].astype(str), validation_frame["label"].astype(str)):
        raise ValueError(f"Seed {seed}: true labels disagree with validation artifact")
    artifact_logits = artifact[[f"logit_{label}" for label in LABELS]].to_numpy(dtype=np.float64)
    artifact_probabilities = artifact[
        [f"prob_{label}" for label in LABELS]
    ].to_numpy(dtype=np.float64)
    if not np.isfinite(artifact_logits).all() or not np.isfinite(artifact_probabilities).all():
        raise ValueError(f"Seed {seed}: non-finite validation artifact values")
    if not np.allclose(artifact_probabilities.sum(axis=1), 1.0, rtol=0, atol=1e-6):
        raise ValueError(f"Seed {seed}: artifact probabilities do not sum to one")
    shifted = artifact_logits - artifact_logits.max(axis=1, keepdims=True)
    probabilities_from_logits = np.exp(shifted)
    probabilities_from_logits /= probabilities_from_logits.sum(axis=1, keepdims=True)
    if not np.allclose(
        artifact_probabilities, probabilities_from_logits, rtol=1e-6, atol=1e-7
    ):
        raise ValueError(f"Seed {seed}: artifact logits/probabilities disagree")
    expected_true_ids = validation_frame["label"].map(LABEL2ID).to_numpy(dtype=np.int64)
    if not np.array_equal(artifact["true_id"].to_numpy(dtype=np.int64), expected_true_ids):
        raise ValueError(f"Seed {seed}: artifact true IDs disagree")
    artifact_predicted_ids = artifact["predicted_id"].to_numpy(dtype=np.int64)
    if not np.array_equal(artifact_logits.argmax(axis=1), artifact_predicted_ids):
        raise ValueError(f"Seed {seed}: artifact predicted IDs disagree with logits")
    expected_predicted_labels = np.asarray(
        [LABELS[index] for index in artifact_predicted_ids], dtype=object
    )
    if not np.array_equal(artifact["predicted_label"].astype(str), expected_predicted_labels):
        raise ValueError(f"Seed {seed}: artifact predicted labels disagree")
    max_abs_diff = float(np.max(np.abs(artifact_logits - reproduced_logits)))
    recorded = summary_row.get("saved_model_reload_logits_max_abs_diff", np.nan)
    recorded = float(recorded) if pd.notna(recorded) else 0.0
    if not np.allclose(
        artifact_logits,
        reproduced_logits,
        rtol=GLOBAL_LOGIT_RTOL,
        atol=GLOBAL_LOGIT_ATOL,
        equal_nan=False,
    ):
        raise ValueError(
            f"Seed {seed}: reproduced logits differ by {max_abs_diff:.3g}; "
            f"required rtol={GLOBAL_LOGIT_RTOL}, atol={GLOBAL_LOGIT_ATOL}"
        )
    reproduced_ids = reproduced_logits.argmax(axis=1)
    if not np.array_equal(reproduced_ids, artifact_predicted_ids):
        raise ValueError(f"Seed {seed}: reproduced validation predictions are not exact")
    return {
        "model_seed": seed,
        "validation_rows": len(artifact),
        "predictions_exact": True,
        "logits_max_abs_diff": max_abs_diff,
        "logits_atol": GLOBAL_LOGIT_ATOL,
        "logits_rtol": GLOBAL_LOGIT_RTOL,
        "upstream_recorded_reload_max_abs_diff": recorded,
        "status": "PASS",
    }


## 7. Linear specialist training and persistence

Checkpoint selection maximizes validation macro-F1, then minimizes validation loss,
then prefers the earlier epoch. Hyperparameters are identical for every seed and pair.


In [7]:
class PairHead(nn.Module):
    def __init__(self, hidden_size: int):
        super().__init__()
        self.classifier = nn.Linear(hidden_size, 2)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.classifier(features)


@torch.inference_mode()
def evaluate_head(head, features, targets, criterion) -> tuple[float, np.ndarray, np.ndarray]:
    head.eval()
    weighted_loss_sum = 0.0
    target_weight_sum = 0.0
    logits_chunks = []
    loader = DataLoader(
        TensorDataset(features, targets),
        batch_size=HEAD_BATCH_SIZE,
        shuffle=False,
    )
    for batch_features, batch_targets in loader:
        batch_features = batch_features.to(DEVICE, non_blocking=True)
        batch_targets = batch_targets.to(DEVICE, non_blocking=True)
        logits = head(batch_features)
        per_example_loss = torch.nn.functional.cross_entropy(
            logits,
            batch_targets,
            weight=criterion.weight,
            reduction="none",
        )
        weighted_loss_sum += per_example_loss.sum().item()
        target_weight_sum += criterion.weight[batch_targets].sum().item()
        logits_chunks.append(logits.float().cpu())
    logits = torch.cat(logits_chunks).numpy()
    predictions = logits.argmax(axis=1)
    return float(weighted_loss_sum / target_weight_sum), logits, predictions


def train_pair_head(
    model_seed: int,
    pair_index: int,
    class_a: str,
    class_b: str,
    train_features: torch.Tensor,
    validation_features: torch.Tensor,
    train_frame: pd.DataFrame,
    validation_frame: pd.DataFrame,
    run_id: str,
    model_manifest_sha256: str,
    artifact_manifest_sha256: str,
    seed_output_dir: Path,
) -> tuple[dict, pd.DataFrame, pd.DataFrame]:
    pair_name = f"{class_a}|{class_b}"
    slug = pair_slug(class_a, class_b)
    head_dir = seed_output_dir / slug
    head_dir.mkdir(parents=True, exist_ok=True)

    train_mask = train_frame["label"].isin([class_a, class_b]).to_numpy()
    validation_mask = validation_frame["label"].isin([class_a, class_b]).to_numpy()
    train_indices = np.flatnonzero(train_mask)
    validation_indices = np.flatnonzero(validation_mask)
    x_train = train_features[train_indices]
    x_validation = validation_features[validation_indices]
    y_train_np = (train_frame.iloc[train_indices]["label"].to_numpy() == class_b).astype(np.int64)
    y_validation_np = (
        validation_frame.iloc[validation_indices]["label"].to_numpy() == class_b
    ).astype(np.int64)
    y_train = torch.from_numpy(y_train_np)
    y_validation = torch.from_numpy(y_validation_np)

    class_weights = compute_class_weights(y_train_np)
    head_seed = deterministic_head_seed(model_seed, pair_index)
    seed_everything(head_seed)
    head = PairHead(train_features.shape[1]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    optimizer = torch.optim.AdamW(
        head.parameters(), lr=HEAD_LR, weight_decay=HEAD_WEIGHT_DECAY
    )
    generator = torch.Generator().manual_seed(head_seed)
    loader = DataLoader(
        TensorDataset(x_train, y_train),
        batch_size=HEAD_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        pin_memory=True,
    )

    best_key = None
    best_state = None
    best_epoch = None
    best_train_loss = None
    history_rows = []
    stale_epochs = 0

    for epoch in range(1, HEAD_MAX_EPOCHS + 1):
        head.train()
        epoch_loss_sum = 0.0
        epoch_weight_sum = 0.0
        for batch_features, batch_targets in loader:
            batch_features = batch_features.to(DEVICE, non_blocking=True)
            batch_targets = batch_targets.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = head(batch_features)
            loss = criterion(logits, batch_targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(head.parameters(), HEAD_MAX_GRAD_NORM)
            optimizer.step()
            per_example_loss = torch.nn.functional.cross_entropy(
                logits.detach(),
                batch_targets,
                weight=criterion.weight,
                reduction="none",
            )
            epoch_loss_sum += per_example_loss.sum().item()
            epoch_weight_sum += criterion.weight[batch_targets].sum().item()
        train_loss = float(epoch_loss_sum / epoch_weight_sum)
        validation_loss, validation_logits, validation_predictions = evaluate_head(
            head, x_validation, y_validation, criterion
        )
        metrics = binary_metrics(y_validation_np, validation_predictions)
        history_rows.append(
            {
                "model_seed": model_seed,
                "head_seed": head_seed,
                "pair_index": pair_index,
                "pair": pair_name,
                "epoch": epoch,
                "train_loss": train_loss,
                "validation_loss": validation_loss,
                "validation_macro_f1": metrics["macro_f1"],
                "validation_accuracy": metrics["accuracy"],
            }
        )
        candidate_key = selection_key(metrics["macro_f1"], validation_loss, epoch)
        if best_key is None or candidate_key < best_key:
            best_key = candidate_key
            best_state = {
                name: tensor.detach().cpu().clone() for name, tensor in head.state_dict().items()
            }
            best_epoch = epoch
            best_train_loss = train_loss
            stale_epochs = 0
        else:
            stale_epochs += 1
        if stale_epochs >= HEAD_PATIENCE:
            break

    if best_state is None:
        raise RuntimeError(f"No checkpoint selected for seed {model_seed}, pair {pair_name}")
    head.load_state_dict(best_state)
    best_validation_loss, best_logits, best_predictions = evaluate_head(
        head, x_validation, y_validation, criterion
    )
    best_metrics = binary_metrics(y_validation_np, best_predictions)

    head_path = head_dir / "head.safetensors"
    temporary_head_path = head_dir / "head.tmp.safetensors"
    save_file(best_state, str(temporary_head_path))
    temporary_head_path.replace(head_path)

    reloaded = PairHead(train_features.shape[1]).to(DEVICE)
    reloaded.load_state_dict(load_file(str(head_path), device="cpu"))
    reload_loss, reload_logits, reload_predictions = evaluate_head(
        reloaded, x_validation, y_validation, criterion
    )
    if not np.array_equal(best_predictions, reload_predictions):
        raise ValueError(f"Saved-head prediction reload mismatch: {model_seed} {pair_name}")
    reload_max_abs_diff = float(np.max(np.abs(best_logits - reload_logits)))
    if reload_max_abs_diff > 1e-7 or abs(reload_loss - best_validation_loss) > 1e-10:
        raise ValueError(f"Saved-head logit reload mismatch: {model_seed} {pair_name}")

    probabilities = torch.softmax(torch.from_numpy(best_logits), dim=1).numpy()
    validation_rows = validation_frame.iloc[validation_indices]
    prediction_frame = pd.DataFrame(
        {
            "model_seed": model_seed,
            "head_seed": head_seed,
            "pair_index": pair_index,
            "pair": pair_name,
            "class_a": class_a,
            "class_b": class_b,
            "original_row": validation_rows["original_row"].to_numpy(),
            "true_label": validation_rows["label"].to_numpy(),
            "true_binary_id": y_validation_np,
            "predicted_binary_id": best_predictions,
            "predicted_label": np.where(best_predictions == 0, class_a, class_b),
            "logit_class_a": best_logits[:, 0],
            "logit_class_b": best_logits[:, 1],
            "prob_class_a": probabilities[:, 0],
            "prob_class_b": probabilities[:, 1],
        }
    )
    history_frame = pd.DataFrame(history_rows)
    metric_record = {
        "model_seed": model_seed,
        "head_seed": head_seed,
        "run_id": run_id,
        "encoder_model_manifest_sha256": model_manifest_sha256,
        "encoder_artifact_manifest_sha256": artifact_manifest_sha256,
        "pair_index": pair_index,
        "pair": pair_name,
        "class_a": class_a,
        "class_b": class_b,
        "hidden_size": int(train_features.shape[1]),
        "train_support": int(len(y_train_np)),
        "validation_support": int(len(y_validation_np)),
        "train_class_a_support": int(np.sum(y_train_np == 0)),
        "train_class_b_support": int(np.sum(y_train_np == 1)),
        "class_a_weight": float(class_weights[0]),
        "class_b_weight": float(class_weights[1]),
        "best_epoch": int(best_epoch),
        "epochs_trained": int(len(history_frame)),
        "best_train_loss": float(best_train_loss),
        "validation_loss": float(best_validation_loss),
        **best_metrics,
        "saved_head_predictions_exact_after_reload": True,
        "saved_head_logits_max_abs_diff_after_reload": reload_max_abs_diff,
        "head_sha256": sha256_file(head_path),
        "status": "PASS",
    }
    metadata = {
        **metric_record,
        "phase": 8,
        "architecture": "Linear(hidden_size, 2)",
        "representation": "final_hidden_state_token_0",
        "encoder_frozen": True,
        "class_mapping": {class_a: 0, class_b: 1},
        "loss": "class_weighted_cross_entropy",
        "class_weight_formula": "N / (2 * N_c)",
        "optimizer": "AdamW",
        "learning_rate": HEAD_LR,
        "weight_decay": HEAD_WEIGHT_DECAY,
        "batch_size": HEAD_BATCH_SIZE,
        "maximum_epochs": HEAD_MAX_EPOCHS,
        "patience": HEAD_PATIENCE,
        "maximum_gradient_norm": HEAD_MAX_GRAD_NORM,
        "train_sha256": TRAIN_SHA256,
        "validation_sha256": VALIDATION_SHA256,
        "test_accessed": False,
        "protocol_fingerprint": PROTOCOL_FINGERPRINT,
    }
    write_json(head_dir / "metadata.json", metadata)
    prediction_frame.to_csv(head_dir / "validation_predictions.csv", index=False)
    history_frame.to_csv(head_dir / "training_history.csv", index=False)
    write_json(head_dir / "metrics.json", metric_record)

    del head, reloaded, criterion, optimizer
    torch.cuda.empty_cache()
    return metric_record, prediction_frame, history_frame


def seed_file_hashes(seed_output_dir: Path) -> dict[str, str]:
    return {
        path.relative_to(seed_output_dir).as_posix(): sha256_file(path)
        for path in sorted(item for item in seed_output_dir.rglob("*") if item.is_file())
        if path.name != "seed_complete.json"
    }


def load_completed_seed(seed: int, seed_output_dir: Path, run: dict):
    marker = seed_output_dir / "seed_complete.json"
    metrics_path = seed_output_dir / "seed_head_metrics.csv"
    predictions_path = seed_output_dir / "seed_validation_predictions.csv"
    history_path = seed_output_dir / "seed_training_history.csv"
    if not all(path.is_file() for path in [marker, metrics_path, predictions_path, history_path]):
        return None
    marker_data = json.loads(marker.read_text(encoding="utf-8"))
    expected_hashes = marker_data.get("files_sha256", {})
    expected_seed_files_without_marker = expected_seed_relative_paths() - {"seed_complete.json"}
    if (
        not expected_hashes
        or set(expected_hashes) != expected_seed_files_without_marker
        or seed_file_hashes(seed_output_dir) != expected_hashes
    ):
        return None
    metrics = pd.read_csv(metrics_path)
    predictions = pd.read_csv(predictions_path)
    history = pd.read_csv(history_path)
    expected_pair_names = {f"{a}|{b}" for a, b in ALL_PAIRS}
    if (
        marker_data.get("status") != "PASS"
        or int(marker_data.get("model_seed", -1)) != seed
        or marker_data.get("run_id") != run["run_id"]
        or marker_data.get("model_sha256") != run["model_sha256"]
        or marker_data.get("model_manifest_sha256") != run["model_manifest_sha256"]
        or marker_data.get("artifact_manifest_sha256") != run["artifact_manifest_sha256"]
        or marker_data.get("protocol_fingerprint") != PROTOCOL_FINGERPRINT
        or marker_data.get("train_sha256") != TRAIN_SHA256
        or marker_data.get("validation_sha256") != VALIDATION_SHA256
        or len(metrics) != 10
        or set(metrics["pair"]) != expected_pair_names
        or not metrics["status"].eq("PASS").all()
    ):
        return None
    if not metrics["model_seed"].astype(int).eq(seed).all():
        return None
    required_prediction_columns = {
        "model_seed", "pair", "original_row", "true_label", "true_binary_id",
        "predicted_binary_id", "prob_class_a", "prob_class_b",
        "logit_class_a", "logit_class_b",
    }
    required_history_columns = {
        "model_seed", "pair", "epoch", "train_loss", "validation_loss",
        "validation_macro_f1", "validation_accuracy",
    }
    if not required_prediction_columns.issubset(predictions.columns):
        return None
    if not required_history_columns.issubset(history.columns):
        return None
    if predictions.empty or history.empty:
        return None
    prediction_numeric = predictions[
        ["true_binary_id", "predicted_binary_id", "prob_class_a", "prob_class_b",
         "logit_class_a", "logit_class_b"]
    ].to_numpy(dtype=float)
    history_numeric = history[
        ["epoch", "train_loss", "validation_loss", "validation_macro_f1", "validation_accuracy"]
    ].to_numpy(dtype=float)
    if not np.isfinite(prediction_numeric).all() or not np.isfinite(history_numeric).all():
        return None
    if not np.allclose(
        predictions["prob_class_a"] + predictions["prob_class_b"], 1.0, rtol=0, atol=1e-6
    ):
        return None
    if predictions[["model_seed", "pair", "original_row"]].duplicated().any():
        return None
    for class_a, class_b in ALL_PAIRS:
        head_dir = seed_output_dir / pair_slug(class_a, class_b)
        if not (head_dir / "head.safetensors").is_file() or not (
            head_dir / "metadata.json"
        ).is_file():
            return None
        metadata = json.loads((head_dir / "metadata.json").read_text(encoding="utf-8"))
        pair_name = f"{class_a}|{class_b}"
        metric_row = metrics.loc[metrics["pair"] == pair_name]
        pair_predictions = predictions.loc[predictions["pair"] == pair_name]
        pair_history = history.loc[history["pair"] == pair_name]
        if (
            len(metric_row) != 1
            or metadata.get("protocol_fingerprint") != PROTOCOL_FINGERPRINT
            or metadata.get("pair") != pair_name
            or int(metadata.get("model_seed", -1)) != seed
            or len(pair_predictions) != int(metric_row.iloc[0]["validation_support"])
            or len(pair_history) != int(metric_row.iloc[0]["epochs_trained"])
            or set(pair_predictions["true_label"]) != {class_a, class_b}
        ):
            return None
        tensors = load_file(str(head_dir / "head.safetensors"), device="cpu")
        if set(tensors) != {"classifier.weight", "classifier.bias"}:
            return None
        if tensors["classifier.weight"].shape[0] != 2 or tensors["classifier.bias"].shape != (2,):
            return None
        if not all(torch.isfinite(tensor).all() for tensor in tensors.values()):
            return None
    return metrics, predictions, history, marker_data["global_reproduction"]


# Synthetic persistence and corruption-detection checks before scientific training.
with tempfile.TemporaryDirectory() as _tmp:
    _tmp = Path(_tmp)
    seed_everything(12345)
    _toy_features = torch.arange(20, dtype=torch.float32, device=DEVICE).reshape(5, 4)
    _toy_head = PairHead(4).to(DEVICE).eval()
    with torch.inference_mode():
        _toy_logits_before = _toy_head(_toy_features).cpu()
    _toy_path = _tmp / "head.safetensors"
    save_file(
        {name: tensor.detach().cpu().contiguous() for name, tensor in _toy_head.state_dict().items()},
        str(_toy_path),
    )
    _toy_reloaded = PairHead(4).to(DEVICE).eval()
    _toy_reloaded.load_state_dict(load_file(str(_toy_path), device="cpu"))
    with torch.inference_mode():
        _toy_logits_after = _toy_reloaded(_toy_features).cpu()
    assert torch.equal(_toy_logits_before, _toy_logits_after)

    _resume_dir = _tmp / "resume"
    _resume_dir.mkdir()
    (_resume_dir / "payload.csv").write_text("a\n1\n", encoding="utf-8")
    _before_corruption = seed_file_hashes(_resume_dir)
    (_resume_dir / "payload.csv").write_text("a\n2\n", encoding="utf-8")
    assert seed_file_hashes(_resume_dir) != _before_corruption
del _toy_head, _toy_reloaded, _toy_features, _toy_logits_before, _toy_logits_after
torch.cuda.empty_cache()
print("PASS: saved-head reload and resume-corruption self-tests")


PASS: saved-head reload and resume-corruption self-tests


## 8. Train all 100 seed-pair heads

This is the long-running cell. It saves each head immediately and marks a seed
complete only after all ten heads pass reload checks. A rerun can safely skip fully
completed seeds; partial seeds are recomputed.


In [8]:
print("Preflight: reproducing validation outputs for all ten checkpoints before head training")
global_reproduction_rows = []
for seed_position, seed in enumerate(EXPECTED_SEEDS, start=1):
    run = RUNS[seed]
    print(f"  preflight [{seed_position}/10] seed {seed}")
    seed_everything(seed)
    tokenizer, model = load_seed_model(run["model_dir"])
    _, preflight_logits = extract_features_and_logits(
        validation_df["text"].tolist(), tokenizer, model
    )
    summary_row = GLOBAL_SUMMARY.loc[GLOBAL_SUMMARY["seed"].astype(int) == seed].iloc[0]
    reproduction = verify_global_validation_artifact(
        seed,
        validation_df,
        preflight_logits,
        run["artifact_dir"],
        summary_row,
    )
    global_reproduction_rows.append(reproduction)
    del model, tokenizer, preflight_logits
    gc.collect()
    torch.cuda.empty_cache()
if len(global_reproduction_rows) != 10 or not all(
    row["status"] == "PASS" for row in global_reproduction_rows
):
    raise RuntimeError("Global-checkpoint reproduction preflight failed")
reproduction_by_seed = {int(row["model_seed"]): row for row in global_reproduction_rows}
print("PASS: all ten global checkpoints reproduced before specialist outputs")

all_metric_frames = []
all_prediction_frames = []
all_history_frames = []
run_started = time.time()

for seed_position, seed in enumerate(EXPECTED_SEEDS, start=1):
    run = RUNS[seed]
    seed_output_dir = OUTPUT_ROOT / "heads" / f"seed_{seed}"
    completed = (
        load_completed_seed(seed, seed_output_dir, run)
        if RESUME_COMPLETED_SEEDS and seed_output_dir.is_dir()
        else None
    )
    if completed is not None:
        seed_metrics, seed_predictions, seed_history, _ = completed
        all_metric_frames.append(seed_metrics)
        all_prediction_frames.append(seed_predictions)
        all_history_frames.append(seed_history)
        print(f"[{seed_position}/10] seed {seed}: verified completed outputs; skipping")
        continue

    if seed_output_dir.exists():
        if seed_output_dir.parent.resolve() != (OUTPUT_ROOT / "heads").resolve():
            raise RuntimeError(f"Refusing to reset unexpected directory: {seed_output_dir}")
        shutil.rmtree(seed_output_dir)
    seed_output_dir.mkdir(parents=True, exist_ok=False)

    print(f"[{seed_position}/10] seed {seed}: loading {run['run_id']}")
    seed_everything(seed)
    tokenizer, model = load_seed_model(run["model_dir"])
    validation_features, validation_global_logits = extract_features_and_logits(
        validation_df["text"].tolist(), tokenizer, model
    )
    summary_row = GLOBAL_SUMMARY.loc[GLOBAL_SUMMARY["seed"].astype(int) == seed].iloc[0]
    reproduction = reproduction_by_seed[seed]
    train_features, _ = extract_features_and_logits(train_df["text"].tolist(), tokenizer, model)
    if train_features.shape[1] != validation_features.shape[1]:
        raise ValueError(f"Seed {seed}: train/validation hidden sizes disagree")

    del model, tokenizer, validation_global_logits
    gc.collect()
    torch.cuda.empty_cache()

    seed_metric_records = []
    seed_prediction_frames = []
    seed_history_frames = []
    for pair_index, (class_a, class_b) in enumerate(ALL_PAIRS):
        print(f"  head {pair_index + 1}/10: {class_a}|{class_b}")
        metric_record, prediction_frame, history_frame = train_pair_head(
            model_seed=seed,
            pair_index=pair_index,
            class_a=class_a,
            class_b=class_b,
            train_features=train_features,
            validation_features=validation_features,
            train_frame=train_df,
            validation_frame=validation_df,
            run_id=run["run_id"],
            model_manifest_sha256=run["model_manifest_sha256"],
            artifact_manifest_sha256=run["artifact_manifest_sha256"],
            seed_output_dir=seed_output_dir,
        )
        seed_metric_records.append(metric_record)
        seed_prediction_frames.append(prediction_frame)
        seed_history_frames.append(history_frame)

    seed_metrics = pd.DataFrame(seed_metric_records).sort_values("pair_index")
    seed_predictions = pd.concat(seed_prediction_frames, ignore_index=True)
    seed_history = pd.concat(seed_history_frames, ignore_index=True)
    if len(seed_metrics) != 10 or seed_metrics["pair"].nunique() != 10:
        raise ValueError(f"Seed {seed}: incomplete head bank")
    seed_metrics.to_csv(seed_output_dir / "seed_head_metrics.csv", index=False)
    seed_predictions.to_csv(seed_output_dir / "seed_validation_predictions.csv", index=False)
    seed_history.to_csv(seed_output_dir / "seed_training_history.csv", index=False)
    completed_file_hashes = seed_file_hashes(seed_output_dir)
    write_json(
        seed_output_dir / "seed_complete.json",
        {
            "phase": 8,
            "model_seed": seed,
            "run_id": run["run_id"],
            "model_sha256": run["model_sha256"],
            "model_manifest_sha256": run["model_manifest_sha256"],
            "artifact_manifest_sha256": run["artifact_manifest_sha256"],
            "head_count": 10,
            "pairs": [f"{a}|{b}" for a, b in ALL_PAIRS],
            "global_reproduction": reproduction,
            "protocol_fingerprint": PROTOCOL_FINGERPRINT,
            "train_sha256": TRAIN_SHA256,
            "validation_sha256": VALIDATION_SHA256,
            "files_sha256": completed_file_hashes,
            "status": "PASS",
            "test_accessed": False,
        },
    )
    all_metric_frames.append(seed_metrics)
    all_prediction_frames.append(seed_predictions)
    all_history_frames.append(seed_history)

    del train_features, validation_features
    gc.collect()
    torch.cuda.empty_cache()

elapsed_seconds = time.time() - run_started
print(f"All seed loops finished in {elapsed_seconds / 3600:.2f} hours")


Preflight: reproducing validation outputs for all ten checkpoints before head training
  preflight [1/10] seed 13


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  preflight [2/10] seed 42


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  preflight [3/10] seed 77


...

  head 1/10: Suicidal|Depressed
  head 2/10: Suicidal|Anxious
  head 3/10: Suicidal|Frustrated
  head 4/10: Suicidal|Others
  head 5/10: Depressed|Anxious
  head 6/10: Depressed|Frustrated
  head 7/10: Depressed|Others
  head 8/10: Anxious|Frustrated
  head 9/10: Anxious|Others
  head 10/10: Frustrated|Others
[9/10] seed 2026: loading mentalroberta_full_seed2026_len512_f6630d942394


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  head 1/10: Suicidal|Depressed
  head 2/10: Suicidal|Anxious
  head 3/10: Suicidal|Frustrated
  head 4/10: Suicidal|Others
  head 5/10: Depressed|Anxious
  head 6/10: Depressed|Frustrated
  head 7/10: Depressed|Others
  head 8/10: Anxious|Frustrated
  head 9/10: Anxious|Others
  head 10/10: Frustrated|Others
[10/10] seed 3407: loading mentalroberta_full_seed3407_len512_d8596ea710c3


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  head 1/10: Suicidal|Depressed
  head 2/10: Suicidal|Anxious
  head 3/10: Suicidal|Frustrated
  head 4/10: Suicidal|Others
  head 5/10: Depressed|Anxious
  head 6/10: Depressed|Frustrated
  head 7/10: Depressed|Others
  head 8/10: Anxious|Frustrated
  head 9/10: Anxious|Others
  head 10/10: Frustrated|Others
All seed loops finished in 0.54 hours


## 9. Aggregate, audit, report, and package

Completion is accepted only if every one of the 100 seed-pair combinations exists,
probabilities are valid, saved heads and metadata exist, and all output hashes verify.


In [9]:
metrics_df = pd.concat(all_metric_frames, ignore_index=True).sort_values(
    ["model_seed", "pair_index"]
).reset_index(drop=True)
predictions_df = pd.concat(all_prediction_frames, ignore_index=True).sort_values(
    ["model_seed", "pair_index", "original_row"]
).reset_index(drop=True)
history_df = pd.concat(all_history_frames, ignore_index=True).sort_values(
    ["model_seed", "pair_index", "epoch"]
).reset_index(drop=True)
reproduction_df = pd.DataFrame(global_reproduction_rows).sort_values("model_seed")

expected_combinations = expected_seed_pair_combinations()
observed_combinations = set(zip(metrics_df["model_seed"].astype(int), metrics_df["pair"]))
assert len(metrics_df) == 100
assert metrics_df[["model_seed", "pair"]].drop_duplicates().shape[0] == 100
assert observed_combinations == expected_combinations
assert metrics_df["status"].eq("PASS").all()
assert metrics_df["saved_head_predictions_exact_after_reload"].astype(bool).all()
assert sorted(reproduction_df["model_seed"].astype(int)) == sorted(EXPECTED_SEEDS)
assert reproduction_df["predictions_exact"].astype(bool).all()

numeric_metric_columns = metrics_df.select_dtypes(include=[np.number]).columns
if not np.isfinite(metrics_df[numeric_metric_columns].to_numpy()).all():
    raise ValueError("Non-finite aggregate head metric")
probability_sums = predictions_df["prob_class_a"] + predictions_df["prob_class_b"]
if not np.allclose(probability_sums, 1.0, rtol=0, atol=1e-6):
    raise ValueError("Specialist probabilities do not sum to one")
if not predictions_df[["logit_class_a", "logit_class_b", "prob_class_a", "prob_class_b"]].apply(
    np.isfinite
).all().all():
    raise ValueError("Non-finite specialist prediction value")
for metric_row in metrics_df.itertuples(index=False):
    key_mask = (
        (predictions_df["model_seed"].astype(int) == int(metric_row.model_seed))
        & (predictions_df["pair"] == metric_row.pair)
    )
    pair_predictions = predictions_df.loc[key_mask]
    history_mask = (
        (history_df["model_seed"].astype(int) == int(metric_row.model_seed))
        & (history_df["pair"] == metric_row.pair)
    )
    pair_history = history_df.loc[history_mask].sort_values("epoch")
    expected_validation_ids = set(
        validation_df.loc[
            validation_df["label"].isin([metric_row.class_a, metric_row.class_b]),
            "original_row",
        ]
    )
    if len(pair_predictions) != int(metric_row.validation_support):
        raise ValueError(f"Prediction coverage mismatch: {metric_row.model_seed} {metric_row.pair}")
    if set(pair_predictions["original_row"]) != expected_validation_ids:
        raise ValueError(f"Prediction row IDs mismatch: {metric_row.model_seed} {metric_row.pair}")
    if set(pair_predictions["true_label"]) != {metric_row.class_a, metric_row.class_b}:
        raise ValueError(f"Prediction true-label mismatch: {metric_row.model_seed} {metric_row.pair}")
    if len(pair_history) != int(metric_row.epochs_trained):
        raise ValueError(f"History coverage mismatch: {metric_row.model_seed} {metric_row.pair}")
    if pair_history["epoch"].astype(int).tolist() != list(
        range(1, int(metric_row.epochs_trained) + 1)
    ):
        raise ValueError(f"History epochs mismatch: {metric_row.model_seed} {metric_row.pair}")

completeness_rows = []
for seed in EXPECTED_SEEDS:
    for pair_index, (class_a, class_b) in enumerate(ALL_PAIRS):
        pair_name = f"{class_a}|{class_b}"
        head_dir = OUTPUT_ROOT / "heads" / f"seed_{seed}" / pair_slug(class_a, class_b)
        head_path = head_dir / "head.safetensors"
        metadata_path = head_dir / "metadata.json"
        if not head_path.is_file() or not metadata_path.is_file():
            raise FileNotFoundError(f"Missing saved head files: seed {seed}, {pair_name}")
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if metadata.get("pair") != pair_name or int(metadata.get("model_seed", -1)) != seed:
            raise ValueError(f"Saved head metadata mismatch: seed {seed}, {pair_name}")
        if metadata.get("protocol_fingerprint") != PROTOCOL_FINGERPRINT:
            raise ValueError(f"Saved head protocol mismatch: seed {seed}, {pair_name}")
        tensors = load_file(str(head_path), device="cpu")
        if set(tensors) != {"classifier.weight", "classifier.bias"}:
            raise ValueError(f"Unexpected tensor keys: seed {seed}, {pair_name}")
        if tensors["classifier.weight"].shape != (2, int(metadata["hidden_size"])):
            raise ValueError(f"Unexpected head weight shape: seed {seed}, {pair_name}")
        if tensors["classifier.bias"].shape != (2,):
            raise ValueError(f"Unexpected head bias shape: seed {seed}, {pair_name}")
        if not all(torch.isfinite(tensor).all() for tensor in tensors.values()):
            raise ValueError(f"Non-finite saved tensor: seed {seed}, {pair_name}")
        if sha256_file(head_path) != metadata.get("head_sha256"):
            raise ValueError(f"Saved head hash mismatch: seed {seed}, {pair_name}")
        completeness_rows.append(
            {
                "model_seed": seed,
                "pair_index": pair_index,
                "pair": pair_name,
                "head_file": head_path.relative_to(OUTPUT_ROOT).as_posix(),
                "head_sha256": sha256_file(head_path),
                "metadata_file": metadata_path.relative_to(OUTPUT_ROOT).as_posix(),
                "complete": True,
            }
        )
completeness_df = pd.DataFrame(completeness_rows)
if len(completeness_df) != 100 or not completeness_df["complete"].all():
    raise ValueError("100-head completeness audit failed")

metrics_df.to_csv(OUTPUT_ROOT / "phase8_head_metrics.csv", index=False)
predictions_df.to_csv(OUTPUT_ROOT / "phase8_validation_predictions.csv", index=False)
history_df.to_csv(OUTPUT_ROOT / "phase8_training_history.csv", index=False)
reproduction_df.to_csv(OUTPUT_ROOT / "phase8_global_reproduction.csv", index=False)
completeness_df.to_csv(OUTPUT_ROOT / "phase8_completeness_matrix.csv", index=False)

pair_summary = (
    metrics_df.groupby(["pair_index", "pair", "class_a", "class_b"], as_index=False)
    .agg(
        seed_count=("model_seed", "nunique"),
        mean_validation_macro_f1=("macro_f1", "mean"),
        sample_sd_validation_macro_f1=("macro_f1", "std"),
        min_validation_macro_f1=("macro_f1", "min"),
        max_validation_macro_f1=("macro_f1", "max"),
        mean_validation_accuracy=("accuracy", "mean"),
        mean_balanced_accuracy=("balanced_accuracy", "mean"),
        median_best_epoch=("best_epoch", "median"),
    )
    .sort_values("pair_index")
)
seed_summary = (
    metrics_df.groupby("model_seed", as_index=False)
    .agg(
        head_count=("pair", "nunique"),
        mean_validation_macro_f1=("macro_f1", "mean"),
        sample_sd_validation_macro_f1=("macro_f1", "std"),
        min_validation_macro_f1=("macro_f1", "min"),
        max_validation_macro_f1=("macro_f1", "max"),
        mean_validation_accuracy=("accuracy", "mean"),
    )
    .sort_values("model_seed")
)
if len(pair_summary) != 10 or not pair_summary["seed_count"].eq(10).all():
    raise ValueError("Pair summary completeness failed")
if len(seed_summary) != 10 or not seed_summary["head_count"].eq(10).all():
    raise ValueError("Seed summary completeness failed")
pair_summary.to_csv(OUTPUT_ROOT / "phase8_pair_summary.csv", index=False)
seed_summary.to_csv(OUTPUT_ROOT / "phase8_seed_summary.csv", index=False)

protocol = {
    "phase": 8,
    "title": "All-Pair Frozen-Encoder Specialists",
    "execution_device": "CUDA GPU",
    "float_policy": "float32 encoder inference and head training",
    "model_seeds": EXPECTED_SEEDS,
    "labels": LABELS,
    "pairs": [f"{a}|{b}" for a, b in ALL_PAIRS],
    "head_count": 100,
    "representation": "final_hidden_state_token_0",
    "encoder_frozen": True,
    "head_architecture": "Linear(hidden_size, 2)",
    "loss": "class_weighted_cross_entropy",
    "class_weight_formula": "N / (2 * N_c)",
    "optimizer": "AdamW",
    "learning_rate": HEAD_LR,
    "weight_decay": HEAD_WEIGHT_DECAY,
    "batch_size": HEAD_BATCH_SIZE,
    "maximum_epochs": HEAD_MAX_EPOCHS,
    "early_stopping_patience": HEAD_PATIENCE,
    "selection_rule": [
        "maximum validation macro-F1",
        "lower validation loss",
        "earlier epoch",
    ],
    "phase5_frozen_H": PHASE5_SELECTION["H"],
    "phase5_frozen_K": PHASE5_SELECTION["K"],
    "routing_performed": False,
    "tau_selected": False,
    "test_accessed": False,
    "train_sha256": TRAIN_SHA256,
    "validation_sha256": VALIDATION_SHA256,
    "protocol_fingerprint": PROTOCOL_FINGERPRINT,
}
write_json(OUTPUT_ROOT / "phase8_protocol.json", protocol)

report_lines = [
    "# Phase 8 All-Pair Specialist Report",
    "",
    "## Material Passport",
    "",
    "- Data scope: locked training and validation splits only.",
    "- Global models: ten verified, seed-matched MentalRoBERTa checkpoints.",
    "- Specialist bank: 10 seeds × 10 unordered pairs = 100 linear heads.",
    "- Representation: frozen final hidden-layer token-zero (`<s>`) vector.",
    "- Loss: class-weighted cross-entropy with `N / (2 * N_c)`.",
    "- Execution: CUDA GPU; one encoder resident at a time; float32 inference.",
    "- Test access: false.",
    "- Routing: not performed; no `tau` was selected.",
    "",
    "## Completeness",
    "",
    "All 100 seed-pair heads passed persistence and reload verification.",
    "Each seed contributes all ten pairs and each pair contributes all ten seeds.",
    "",
    "## Frozen downstream routing scope",
    "",
    f"Phase 5 remains frozen at `K={PHASE5_SELECTION['K']}` with:",
]
report_lines.extend(f"- `{pair}`" for pair in PHASE5_SELECTION["H"])
report_lines.extend(
    [
        "",
        "Phase 9 may activate only those three pair types and calibrate `tau` on validation data.",
        "Phase 8 neither routes examples nor changes the frozen hard-pair set.",
        "",
        "## Aggregate validation summary",
        "",
        "```text",
        pair_summary.to_string(index=False, float_format=lambda value: f"{value:.6f}"),
        "```",
        "",
        "These are validation descriptions of frozen-encoder specialists, not test estimates.",
    ]
)
(OUTPUT_ROOT / "phase8_report.md").write_text("\n".join(report_lines) + "\n", encoding="utf-8")

execution_summary = {
    "phase": 8,
    "status": "PASS",
    "model_seed_count": 10,
    "pair_count": 10,
    "head_count": 100,
    "prediction_rows": len(predictions_df),
    "history_rows": len(history_df),
    "global_reproduction_all_pass": True,
    "saved_head_reload_all_pass": True,
    "phase5_frozen_H": PHASE5_SELECTION["H"],
    "phase5_frozen_K": PHASE5_SELECTION["K"],
    "routing_performed": False,
    "tau_selected": False,
    "test_accessed": False,
    "protocol_fingerprint": PROTOCOL_FINGERPRINT,
    "result_zip": ZIP_PATH.name,
}
write_json(OUTPUT_ROOT / "phase8_execution_summary.json", execution_summary)

old_output_manifest = OUTPUT_ROOT / "sha256_manifest.csv"
if old_output_manifest.exists():
    old_output_manifest.unlink()
actual_before_manifest = {
    path.relative_to(OUTPUT_ROOT).as_posix()
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
}
expected_before_manifest = expected_output_relative_paths(include_manifest=False)
if actual_before_manifest != expected_before_manifest:
    raise ValueError(
        "Output-tree allowlist mismatch before packaging; "
        f"missing={sorted(expected_before_manifest - actual_before_manifest)[:10]}, "
        f"unexpected={sorted(actual_before_manifest - expected_before_manifest)[:10]}"
    )
manifest_df = write_output_manifest(OUTPUT_ROOT)
verified_output_manifest = verify_manifest(OUTPUT_ROOT / "sha256_manifest.csv")
actual_after_manifest = {
    path.relative_to(OUTPUT_ROOT).as_posix()
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
}
if actual_after_manifest != expected_output_relative_paths(include_manifest=True):
    raise ValueError("Output-tree allowlist mismatch after manifest creation")
zip_sha256 = build_deterministic_zip(OUTPUT_ROOT, ZIP_PATH)
ZIP_HASH_PATH.write_text(f"{zip_sha256}  {ZIP_PATH.name}\n", encoding="utf-8")

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    archive.testzip_result = archive.testzip()
    if archive.testzip_result is not None:
        raise ValueError(f"Corrupt ZIP member: {archive.testzip_result}")
    expected_names = {
        (Path(OUTPUT_ROOT.name) / path.relative_to(OUTPUT_ROOT)).as_posix()
        for path in OUTPUT_ROOT.rglob("*")
        if path.is_file()
    }
    if set(archive.namelist()) != expected_names:
        raise ValueError("ZIP contents differ from the declared Phase 8 output tree")

print("PASS: Phase 8 complete")
print("Heads:", len(metrics_df))
print("Output manifest files:", len(manifest_df))
print("ZIP:", ZIP_PATH)
print("ZIP SHA-256:", zip_sha256)
display(pair_summary)
display(seed_summary)


PASS: Phase 8 complete
Heads: 100
Output manifest files: 551
ZIP: /kaggle/working/MentalDistress_Phase8_All_Pair_Specialists_Results.zip
ZIP SHA-256: 6d30115f9da6a1963b2fc74eb1b4c057ecb685ca7ae6dcf0c61d177bb3fa89e1


,pair_index,pair,class_a,class_b,seed_count,mean_validation_macro_f1,sample_sd_validation_macro_f1,min_validation_macro_f1,max_validation_macro_f1,mean_validation_accuracy,mean_balanced_accuracy,median_best_epoch
0,0,Suicidal|Depressed,Suicidal,Depressed,10,0.915994,0.003724,0.909578,0.920143,0.919145,0.914784,2.0
1,1,Suicidal|Anxious,Suicidal,Anxious,10,0.981530,0.002786,0.978742,0.985296,0.981566,0.981409,3.0
2,2,Suicidal|Frustrated,Suicidal,Frustrated,10,0.967423,0.006197,0.952660,0.973045,0.967617,0.967064,1.0
3,3,Suicidal|Others,Suicidal,Others,10,0.964896,0.006897,0.946961,0.971203,0.965143,0.965224,4.0
4,4,Depressed|Anxious,Depressed,Anxious,10,0.959205,0.011184,0.932029,0.972146,0.960039,0.959952,4.5
5,5,Depressed|Frustrated,Depressed,Frustrated,10,0.959054,0.008014,0.944426,0.969409,0.959677,0.958640,3.0
6,6,Depressed|Others,Depressed,Others,10,0.933214,0.011354,0.911573,0.954555,0.938167,0.932382,1.0
7,7,Anxious|Frustrated,Anxious,Frustrated,10,0.983169,0.005120,0.971907,0.989483,0.983187,0.983162,1.0
8,8,Anxious|Others,Anxious,Others,10,0.960867,0.007471,0.944443,0.969915,0.961481,0.961219,1.5
9,9,Frustrated|Others,Frustrated,Others,10,0.896698,0.008240,0.883268,0.907302,0.899240,0.896930,2.0


,model_seed,head_count,mean_validation_macro_f1,sample_sd_validation_macro_f1,min_validation_macro_f1,max_validation_macro_f1,mean_validation_accuracy
0,13,10,0.958000,0.027241,0.905148,0.989483,0.959085
1,42,10,0.955496,0.029363,0.897991,0.987733,0.956748
2,77,10,0.956642,0.029026,0.896191,0.980716,0.957750
3,123,10,0.945638,0.029655,0.887313,0.985296,0.947269
4,211,10,0.953068,0.027303,0.906746,0.985982,0.954311
5,340,10,0.953789,0.025276,0.907302,0.985984,0.955150
6,512,10,0.954433,0.027863,0.898928,0.985977,0.955652
7,1024,10,0.951870,0.032042,0.883268,0.982475,0.953300
8,2026,10,0.942157,0.030021,0.890086,0.985296,0.943579
9,3407,10,0.950956,0.029096,0.894009,0.982467,0.952419


## 10. Files to download

Download both files printed below. The ZIP contains the complete Phase 8 results and
specialist bank; the small `.sha256` file lets you verify the downloaded archive.


In [10]:
final_files = pd.DataFrame(
    [
        {"file": str(ZIP_PATH), "bytes": ZIP_PATH.stat().st_size, "sha256": sha256_file(ZIP_PATH)},
        {
            "file": str(ZIP_HASH_PATH),
            "bytes": ZIP_HASH_PATH.stat().st_size,
            "sha256": sha256_file(ZIP_HASH_PATH),
        },
    ]
)
display(final_files)


,file,bytes,sha256
0,/kaggle/working/MentalDistress_Phase8_All_Pair...,5304508,6d30115f9da6a1963b2fc74eb1b4c057ecb685ca7ae6dc...
1,/kaggle/working/MentalDistress_Phase8_All_Pair...,121,043437f3def4811b6f44535f1c13b7f994b51021f59a50...
